# Multi-Marginal Coupling for a Barycenter

For a quadratic barycenter, a multi-marginal plan chooses tuples $(x_1,x_2,x_3)$ and then maps each tuple to its weighted average
$$
B(x_1,x_2,x_3)=\lambda_1x_1+\lambda_2x_2+\lambda_3x_3.
$$
This tiny one-dimensional example solves the exact multi-marginal linear program and draws how active tuples create barycenter support points.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import ot

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    ORANGE,
    RED,
    VIOLET,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from scipy.optimize import linprog

NAME = "multimarginal-barycenter-coupling"
out = figure_dir(NAME)


The tensor has only $3\times2\times3$ entries, so the linear program is transparent and deterministic.


In [ ]:
x1 = np.array([-2.05, -0.85, 0.35])
x2 = np.array([-1.20, 0.95])
x3 = np.array([-0.35, 1.15, 2.20])
w1 = np.array([0.25, 0.45, 0.30])
w2 = np.array([0.55, 0.45])
w3 = np.array([0.35, 0.40, 0.25])
lam = np.array([0.25, 0.35, 0.40])
shape = (len(x1), len(x2), len(x3))
triples = []
for i, a in enumerate(x1):
    for j, b in enumerate(x2):
        for k, c in enumerate(x3):
            z = lam[0]*a + lam[1]*b + lam[2]*c
            cost = lam[0]*(a-z)**2 + lam[1]*(b-z)**2 + lam[2]*(c-z)**2
            triples.append((i, j, k, z, cost))

nvar = len(triples)
Aeq = []
beq = []
for i in range(len(x1)):
    row = np.zeros(nvar)
    for r, (ii, jj, kk, z, cost) in enumerate(triples):
        if ii == i: row[r] = 1
    Aeq.append(row); beq.append(w1[i])
for j in range(len(x2)):
    row = np.zeros(nvar)
    for r, (ii, jj, kk, z, cost) in enumerate(triples):
        if jj == j: row[r] = 1
    Aeq.append(row); beq.append(w2[j])
for k in range(len(x3)):
    row = np.zeros(nvar)
    for r, (ii, jj, kk, z, cost) in enumerate(triples):
        if kk == k: row[r] = 1
    Aeq.append(row); beq.append(w3[k])
# One marginal constraint is redundant; remove the last one for numerical rank.
res = linprog(np.array([t[-1] for t in triples]), A_eq=np.vstack(Aeq[:-1]), b_eq=np.array(beq[:-1]), bounds=(0, None), method="highs")
if not res.success:
    raise RuntimeError(res.message)
masses = res.x
active = [(triples[r], masses[r]) for r in range(nvar) if masses[r] > 1e-9]
print(len(active), masses.sum())


Active tensor entries are drawn as three colored supports linked to a violet barycenter point.  Disk size is proportional to mass.


In [ ]:
fig, ax = plt.subplots(figsize=(3.05, 2.25))
rows = [2.0, 1.20, 0.40]
bar_y = -0.30
for (i, j, k, z, cost), mass in active:
    pts_x = [x1[i], x2[j], x3[k], z]
    pts_y = [rows[0], rows[1], rows[2], bar_y]
    for s in range(3):
        ax.plot([pts_x[s], z], [pts_y[s], bar_y], color=VIOLET, lw=0.35 + 3.0*np.sqrt(mass), alpha=0.18 + 0.55*mass/masses.max(), zorder=1)
# Supports.
ax.scatter(x1, np.full_like(x1, rows[0]), s=DIRAC_MARKER_SIZE*(0.55 + 1.55*w1/w1.max()), color=RED, marker="o", edgecolor="none", linewidth=0, zorder=3)
ax.scatter(x2, np.full_like(x2, rows[1]), s=DIRAC_MARKER_SIZE*(0.55 + 1.55*w2/w2.max()), color=ORANGE, marker="o", edgecolor="none", linewidth=0, zorder=3)
ax.scatter(x3, np.full_like(x3, rows[2]), s=DIRAC_MARKER_SIZE*(0.55 + 1.55*w3/w3.max()), color=BLUE, marker="o", edgecolor="none", linewidth=0, zorder=3)
# Barycenter atoms aggregated at equal positions if any.
zvals = np.array([t[0][3] for t in active])
zmass = np.array([t[1] for t in active])
ax.scatter(zvals, np.full_like(zvals, bar_y), s=DIRAC_MARKER_SIZE*(0.55 + 2.0*zmass/zmass.max()), color=VIOLET, marker="o", edgecolor="none", linewidth=0, zorder=4)
ax.set_xlim(-2.45, 2.55)
ax.set_ylim(-0.62, 2.32)
remove_axes(ax)
save_pdf(fig, out / "active-tuples.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.75, 1.10))
ax.axhline(0, color=LIGHT_GRAY, lw=0.8, zorder=0)
ax.scatter(zvals, np.zeros_like(zvals), s=DIRAC_MARKER_SIZE*(0.70 + 2.2*zmass/zmass.max()), color=VIOLET, marker="o", edgecolor="none", linewidth=0, zorder=3)
ax.set_xlim(-2.00, 1.75)
ax.set_ylim(-0.28, 0.28)
remove_axes(ax)
save_pdf(fig, out / "barycenter-support.pdf", pad_inches=0.055)
plt.close(fig)
